In [14]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt

raw = yf.download("SPY", start="2025-01-01", end="2026-01-01")
data = raw.droplevel("Ticker", axis=1)
data['daily_return'] = data['Close'].pct_change()

mu = data['daily_return'].mean()
sigma = data['daily_return'].std()


np.random.seed(42)
n_days = 252
n_simulations = 1000

all_paths = []
S0 = data['Close'].iloc[-1]

for i in range(n_simulations):
    random_returns = np.random.normal(mu, sigma, n_days)
    path = [S0]
    for r in random_returns:
        path.append(path[-1] * (1 + r))
    all_paths.append(path)

all_paths = np.array(all_paths)

print(all_paths.shape)

[*********************100%***********************]  1 of 1 completed

(1000, 253)
Last path final price : $1056.91


In [16]:
strike = 700
r_free = 0.05
T = 1

final_prices = all_paths[:, -1]

payoffs = np.maximum(final_prices - strike, 0)

discount_factor = np.exp(-r_free * T)
option_price = payoffs.mean() * discount_factor

print(f"Strike price: ${strike}")
print(f"Current pricee: ${S0:.2f}")
print(f"Option price (MC): ${option_price:.2f}")
print(f"Payoff > 0: {(payoffs > 0).mean():.1%}")

Strike price: $700
Current pricee: $680.06
Option price (MC): $129.98
Payoff > 0: 75.5%


In [22]:
from scipy.stats import norm

d1 = (np.log(S0/strike) #how far current price is from strike (distance)
      + (r_free + 0.5*sigma**2*252)*T) / (sigma*np.sqrt(252)*np.sqrt(T)) 
#expected drift over time period #This is a technical adjustment called Ito's lemma correction. When prices follow a random walk, the average of log returns is slightly less than the arithmetic mean due to compounding. The 0.5σ² term corrects for that. 
#scale by total uncertainty
d2 = d1 - sigma*np.sqrt(252)*np.sqrt(T)
print(f"d1 = {d1:.2f}")
print(f"d2 = {d2:.2f}")

BS_price = S0*norm.cdf(d1) - strike*np.exp(-r_free*T)*norm.cdf(d2)
#BS = (average SPY value you receive when in the money)
#   - (strike price you pay × probability you pay it)
#   = S0 × N(d1)        -    K×e^(-rT) × N(d2)
#   = "expected receipt" -   "expected cost"

print(f"Black-Scholes price: ${BS_price:.2f}")
print(f"Monte Carlo price: ${option_price:.2f}")
print(f"Difference: ${abs(BS_price - option_price):2f}")
                                                        

d1 = 0.21
d2 = 0.01
Black-Scholes price: $59.74
Monte Carlo price: $129.98
Difference: $70.246952


In [19]:
r_daily = r_free / 252
sigma_daily = sigma

np.random.seed(42)
all_paths_rn = []

for i in range(n_simulations):
    random_returns = np.random.normal(r_daily, sigma_daily, n_days)
    path = [S0]
    for r in random_returns:
        path.append(path[-1] * (1 + r))
    all_paths_rn.append(path)

all_paths_rn = np.array(all_paths_rn)
final_prices_rn = all_paths_rn[:, -1]
payoff_rn = np.maximum(final_prices_rn - strike,0)
option_price_rn = payoff_rn.mean() * discount_factor

print(f"MC Risk-neutral price: ${option_price_rn:.2f}")
print(f"Black-Scholes price: ${BS_price:.2f}")
print(f"Difference: ${abs(BS_price - option_price_rn):.2f}")

MC Risk-neutral price: $58.78
Black-Scholes price: $59.74
Difference: $0.95
